**Step 1: Install dependencies**

In [14]:
!pip install transformers torch

**Use Instruction Model**

In [20]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    device=0
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

**Updated function for Mistral**

In [21]:
def ask_health_question(user_input):
    prompt = f"""
    You are a helpful and safe medical assistant.

    Question: {user_input}
    Answer:
    """

    response = chatbot(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return response[0]['generated_text']

In [22]:
while True:
    query = input("Ask a question: ")

    if query.lower() == "exit":
        break

    print(ask_health_question(query))

Ask a question: is paracetamol safe 


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    You are a helpful and safe medical assistant.

    Question: is paracetamol safe 
    Answer:
    
    Paracetamol, also known as acetaminophen, is generally considered safe and effective for pain relief and reducing fever when taken as directed. However, it's important to follow the recommended dosage and not take more than the maximum daily dose of 4000mg for adults.

    If you have any underlying medical conditions or are taking any medications, it's always best to consult with your healthcare provider before taking paracetamol. They can help ensure it's safe for you to use and can provide guidance on the appropriate dosage.

    Additionally, it's important to note that while paracetamol can help relieve pain and reduce fever, it
Ask a question: exit


**Add Safety Layer**

In [23]:
def safety_filter(user_input):
    risky_keywords = [
        "suicide", "kill myself", "overdose", "emergency", "die"
    ]

    for word in risky_keywords:
        if word in user_input.lower():
            return False
    return True

**Clean Response Extraction**

In [24]:
def clean_response(output):
    if "Answer:" in output:
        return output.split("Answer:")[-1].strip()
    return output.strip()

**Combine Everything**

In [25]:
def safe_chatbot(user_input):
    if not safety_filter(user_input):
        return "⚠️ Please seek immediate help from a medical professional."

    raw_output = ask_health_question(user_input)
    return clean_response(raw_output)

**Build Interactive Chat Loop**

In [26]:
def run_chatbot():
    print("🩺 Health Assistant Chatbot")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        response = safe_chatbot(user_input)
        print("Bot:", response)

In [27]:
run_chatbot()

🩺 Health Assistant Chatbot
Type 'exit' to quit

You: What causes headache?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: There can be many causes of headache. Some common ones include:

        - Stress or tension
        - Lack of sleep
        - Dehydration
        - Certain foods or drinks, such as alcohol, caffeine, or chocolate
        - Hormonal changes, such as those that occur during menstruation
        - Medication side effects
        - Infections, such as sinusitis or meningitis
        - Migraines
        - Certain medical conditions, such as high blood pressure or aneurysms

        If you experience frequent or severe headaches, it's important to speak with a healthcare provider to determine the underlying cause and develop an appropriate treatment
You: Is paracetamol safe?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Paracetamol can be safe when used in appropriate doses and for the intended purposes. It is a widely used pain reliever and fever reducer. However, it can be toxic when taken in high doses or combined with other medications that contain certain chemicals. It is important to follow the recommended dosage and to check with your doctor or a pharmacist before taking paracetamol if you have any health conditions or are taking any medications.
You: How to reduce stress?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: 1. Practice relaxation techniques such as deep breathing, meditation, and yoga.
    2. Get regular exercise and eat a healthy diet.
    3. Get enough sleep.
    4. Take breaks from work and other stressful activities.
    5. Talk to a trusted friend or family member about your feelings.
    6. Seek professional help if needed.
You: exit
Goodbye!


**Improve Output Quality**

In [28]:
def ask_health_question(user_input):
    prompt = f"""
    You are a helpful and safe medical assistant.

    Answer in this format:
    - Explanation:
    - Advice:
    - When to see a doctor:

    Question: {user_input}
    Answer:
    """

    response = chatbot(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return response[0]['generated_text']

**Add Memory**

In [29]:
chat_history = []

def chatbot_with_memory(user_input):
    chat_history.append(f"User: {user_input}")

    context = "\n".join(chat_history[-5:])  # last 5 messages

    prompt = f"""
    You are a helpful medical assistant.

    Conversation:
    {context}

    Assistant:
    """

    response = chatbot(prompt, max_new_tokens=150)

    reply = response[0]['generated_text']
    chat_history.append(f"Bot: {reply}")

    return reply